# VEYA v2 · Шаг 5 — экспорт модели для приложения (ONNX)
Превращает обученную нейросеть в один файл `model.onnx`, который работает в API Veya без PyTorch
(ONNX Runtime, ~300 МБ RAM — подходит для бесплатного Render).

* в модель зашиты нормализация и TTA (обычный + зеркальный снимок);
* рядом `model_config.json` — пороги и «зона сомнения» по каждой болезни + честные метрики test / test_ext;
* проверка: ответы ONNX совпадают с PyTorch.

GPU не нужен. Результат: `MyDrive/veya_v2/export/`.

## 0. Подготовка среды
Подключаем Google Drive (там хранятся данные и модели — они не пропадут, если Colab отключится) и записываем код VEYA в сессию.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
os.makedirs('/content/veya_v2', exist_ok=True)
open('/content/veya_v2/__init__.py', 'w').close()
sys.path.insert(0, '/content')
DRIVE = '/content/drive/MyDrive/veya_v2'
os.makedirs(DRIVE, exist_ok=True)

In [ ]:
!pip -q install -U timm onnx onnxruntime

In [ ]:
%%writefile /content/veya_v2/train.py
"""VEYA v2 — обучение базовых моделей ансамбля (multi-task: стадия ДР, ДР, глаукома, катаракта).

Модели («глаза» ансамбля):
  effv2s   — EfficientNetV2-S, 512px  (CNN, ловит мелкие поражения: микроаневризмы, экссудаты)
  convnext — ConvNeXt-Small, 384px    (современная CNN, другие ошибки чем у EfficientNet)
  vit      — ViT-B/14 DINOv2, 224px   (трансформер, смотрит на снимок целиком)
  retfound — RETFound ViT-L, 224px    (foundation-модель, предобучена на 1.6 млн снимков сетчатки;
                                       нужен HF_TOKEN в секретах Colab, иначе используйте vit)

Всё устойчиво к обрывам Colab: чекпоинт на Google Drive каждые N минут, запуск заново — продолжает с места.
"""
from __future__ import annotations

import json
import math
import os
import random
import shutil
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, cohen_kappa_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset, Sampler

try:
    from tqdm.auto import tqdm
except ImportError:  # pragma: no cover
    def tqdm(x, **k):
        return x

BINARY = ["dr", "glaucoma", "cataract"]
HEADS = ["dr_grade"] + BINARY

MODELS = {
    "effv2s": dict(timm="tf_efficientnetv2_s.in21k_ft_in1k", size=512, bs=16, accum=2, lr=3e-4, epochs=14,
                   samples_per_epoch=36000, channels_last=True),
    "convnext": dict(timm="convnext_small.fb_in22k_ft_in1k_384", size=384, bs=16, accum=2, lr=1.5e-4, epochs=12,
                     samples_per_epoch=36000, channels_last=True, drop_path=0.2),
    "vit": dict(timm="vit_base_patch14_dinov2.lvd142m", size=224, bs=32, accum=1, lr=4e-5, epochs=12,
                samples_per_epoch=36000, kwargs=dict(img_size=224), drop_path=0.1, layer_decay=0.75),
    "retfound": dict(timm="vit_large_patch16_224", size=224, bs=16, accum=2, lr=3e-5, epochs=10,
                     samples_per_epoch=36000, kwargs=dict(global_pool="avg"), drop_path=0.2, layer_decay=0.65,
                     hf=("YukunZhou/RETFound_mae_natureCFP", "RETFound_mae_natureCFP.pth")),
}

MEAN = np.array([0.485, 0.456, 0.406], np.float32) * 255
STD = np.array([0.229, 0.224, 0.225], np.float32) * 255


# ============================================================================ data
def load_image(path: str, size: int) -> np.ndarray:
    flag = cv2.IMREAD_REDUCED_COLOR_2 if size <= 256 else cv2.IMREAD_COLOR
    img = cv2.imread(path, flag)
    if img is None:
        img = np.zeros((size, size, 3), np.uint8)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    if img.shape[0] != size:
        img = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    return img


def augment(img: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Аугментации, безопасные для глазного дна (не ломают признаки болезней)."""
    s = img.shape[0]
    if rng.random() < 0.5:
        img = img[:, ::-1]
    if rng.random() < 0.3:
        img = img[::-1]
    ang = rng.uniform(-180, 180)
    scale = rng.uniform(0.92, 1.12)
    tx, ty = rng.uniform(-0.04, 0.04, 2) * s
    M = cv2.getRotationMatrix2D((s / 2, s / 2), ang, scale)
    M[:, 2] += (tx, ty)
    img = cv2.warpAffine(np.ascontiguousarray(img), M, (s, s), flags=cv2.INTER_LINEAR, borderValue=(0, 0, 0))
    img = img.astype(np.float32)
    # яркость / контраст / гамма / баланс каналов (разные камеры и смартфон-насадка)
    img = img * rng.uniform(0.75, 1.25) + rng.uniform(-20, 20)
    mean = img.mean()
    img = (img - mean) * rng.uniform(0.8, 1.25) + mean
    img *= rng.uniform(0.92, 1.08, size=3)[None, None, :]
    img = np.clip(img, 0, 255)
    if rng.random() < 0.3:
        g = rng.uniform(0.8, 1.25)
        img = 255.0 * (img / 255.0) ** g
    if rng.random() < 0.2:
        k = rng.choice([3, 5])
        img = cv2.GaussianBlur(img, (k, k), 0)
    if rng.random() < 0.15:
        img = img + rng.normal(0, rng.uniform(2, 8), img.shape)
    return np.clip(img, 0, 255)


class FundusDS(Dataset):
    def __init__(self, df: pd.DataFrame, root: str, size: int, train: bool):
        self.paths = [os.path.join(root, p) for p in df["path"].values]
        self.y = df[HEADS].values.astype(np.int64)
        self.size = size
        self.train = train

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = load_image(self.paths[i], self.size)
        if self.train:
            rng = np.random.default_rng((os.getpid() * 1_000_003 + i * 7919 + int(time.time() * 1e6)) % 2**32)
            img = augment(img, rng)
        else:
            img = img.astype(np.float32)
        img = ((img - MEAN) / STD).astype(np.float32)
        x = torch.from_numpy(np.ascontiguousarray(img.transpose(2, 0, 1)))
        return x, torch.from_numpy(self.y[i])


class EpochSampler(Sampler):
    """Взвешенная выборка фиксированного размера на эпоху (баланс датасетов и редких болезней).
    Детерминирована по номеру эпохи → после обрыва Colab продолжаем с того же шага."""

    def __init__(self, weights: np.ndarray, n: int, epoch: int, start: int = 0, seed: int = 2026):
        rng = np.random.default_rng(seed + epoch)
        p = weights / weights.sum()
        self.idx = rng.choice(len(weights), size=n, replace=True, p=p)[start:]

    def __iter__(self):
        return iter(self.idx.tolist())

    def __len__(self):
        return len(self.idx)


def sample_weights(df: pd.DataFrame) -> np.ndarray:
    """Вес снимка ~ 1/sqrt(размер его «страты»): страта = датасет + комбинация меток.
    Больные глаукомой/катарактой/тяжёлой ДР попадают в батчи заметно чаще."""
    key = (df["dataset"].astype(str) + "|" + df["dr_grade"].astype(str) + "|" +
           df["glaucoma"].astype(str) + "|" + df["cataract"].astype(str))
    cnt = key.map(key.value_counts())
    w = 1.0 / np.sqrt(cnt.values.astype(np.float64))
    # дополнительно поднимаем редкие позитивы
    w *= np.where(df["glaucoma"].values == 1, 1.5, 1.0)
    w *= np.where(df["cataract"].values == 1, 2.0, 1.0)
    w *= np.where(df["dr_grade"].values >= 3, 1.5, 1.0)
    return w


# ============================================================================ model
class VeyaNet(nn.Module):
    """Backbone + общий многослойный «нейрон-мозг» модели + 4 головы."""

    def __init__(self, name: str, pretrained=True, drop=0.3, drop_path=0.0, **kw):
        super().__init__()
        import timm
        extra = dict(kw)
        if drop_path:
            extra["drop_path_rate"] = drop_path
        self.backbone = timm.create_model(name, pretrained=pretrained, num_classes=0, **extra)
        d = self.backbone.num_features
        self.neck = nn.Sequential(
            nn.Dropout(drop), nn.Linear(d, 768), nn.GELU(),
            nn.Dropout(drop / 2), nn.Linear(768, 384), nn.GELU(),
        )
        self.heads = nn.ModuleDict({
            "dr_grade": nn.Linear(384, 5),
            **{k: nn.Linear(384, 1) for k in BINARY},
        })
        self.emb_dim = d

    def forward(self, x, return_emb=False):
        f = self.backbone(x)
        h = self.neck(f)
        out = {k: (v(h) if k == "dr_grade" else v(h).squeeze(1)) for k, v in self.heads.items()}
        return (out, f) if return_emb else out


def load_retfound(model: VeyaNet, repo: str, fname: str):
    from huggingface_hub import hf_hub_download
    token = os.environ.get("HF_TOKEN")
    path = hf_hub_download(repo, fname, token=token)
    ck = torch.load(path, map_location="cpu", weights_only=False)
    sd = ck.get("model", ck.get("teacher", ck))
    sd = {k.replace("module.", "").replace("backbone.", ""): v for k, v in sd.items()}
    sd = {k: v for k, v in sd.items() if not k.startswith(("decoder", "mask_token", "head"))}
    msg = model.backbone.load_state_dict(sd, strict=False)
    print(f"RETFound загружен. missing={len(msg.missing_keys)} {msg.missing_keys[:6]} "
          f"unexpected={len(msg.unexpected_keys)} {msg.unexpected_keys[:6]}")


def build_model(key: str, pretrained=True) -> VeyaNet:
    cfg = MODELS[key]
    m = VeyaNet(cfg["timm"], pretrained=pretrained and "hf" not in cfg,
                drop_path=cfg.get("drop_path", 0.0), **cfg.get("kwargs", {}))
    if pretrained and "hf" in cfg:
        load_retfound(m, *cfg["hf"])
    return m


def param_groups(model: VeyaNet, lr: float, wd=0.05, layer_decay: float | None = None):
    """Голова учится в 10 раз быстрее; для ViT — послойное затухание lr (нижние слои меняются меньше)."""
    head = list(model.neck.parameters()) + list(model.heads.parameters())
    groups = [dict(params=head, lr=lr * 10, weight_decay=wd)]
    bb = model.backbone
    if layer_decay and hasattr(bb, "blocks"):
        n = len(bb.blocks)
        layers = {}
        for name, p in bb.named_parameters():
            if name.startswith("blocks."):
                lid = int(name.split(".")[1]) + 1
            elif name.startswith(("cls_token", "pos_embed", "patch_embed", "reg_token")):
                lid = 0
            else:
                lid = n + 1
            no_wd = p.ndim == 1 or name.endswith(".bias") or "token" in name or "pos_embed" in name
            layers.setdefault((lid, no_wd), []).append(p)
        for (lid, no_wd), ps in layers.items():
            groups.append(dict(params=ps, lr=lr * layer_decay ** (n + 1 - lid), weight_decay=0.0 if no_wd else wd))
    else:
        dec, nodec = [], []
        for name, p in bb.named_parameters():
            (nodec if p.ndim == 1 or name.endswith(".bias") else dec).append(p)
        groups += [dict(params=dec, lr=lr, weight_decay=wd), dict(params=nodec, lr=lr, weight_decay=0.0)]
    for g in groups:
        g["base_lr"] = g["lr"]
    return groups


# ============================================================================ loss + metrics
def multitask_loss(out, y, pos_weight: dict, smoothing=0.05):
    losses = {}
    yg = y[:, 0]
    m = yg >= 0
    if m.any():
        losses["dr_grade"] = F.cross_entropy(out["dr_grade"][m].float(), yg[m], label_smoothing=smoothing)
    for j, k in enumerate(BINARY, start=1):
        yy = y[:, j]
        m = yy >= 0
        if m.any():
            t = yy[m].float() * (1 - smoothing) + smoothing / 2
            losses[k] = F.binary_cross_entropy_with_logits(
                out[k][m].float(), t, pos_weight=torch.tensor(pos_weight[k], device=t.device))
    total = sum(losses.values()) if losses else out["dr"].sum() * 0
    return total, {k: float(v.detach()) for k, v in losses.items()}


def compute_metrics(logits: dict, y: np.ndarray) -> dict:
    res = {}
    yg = y[:, 0]
    m = yg >= 0
    if m.sum() > 10:
        pred = logits["dr_grade"][m].argmax(1)
        res["dr_grade_acc"] = accuracy_score(yg[m], pred)
        res["dr_grade_qwk"] = cohen_kappa_score(yg[m], pred, weights="quadratic")
    for j, k in enumerate(BINARY, start=1):
        yy = y[:, j]
        m = yy >= 0
        if m.sum() > 10 and 0 < yy[m].sum() < m.sum():
            p = 1 / (1 + np.exp(-logits[k][m]))
            res[f"{k}_auc"] = roc_auc_score(yy[m], p)
            res[f"{k}_bal_acc"] = 0.5 * ((p[yy[m] == 1] >= 0.5).mean() + (p[yy[m] == 0] < 0.5).mean())
    keys = [res.get("dr_auc"), res.get("glaucoma_auc"), res.get("cataract_auc"), res.get("dr_grade_qwk")]
    keys = [k for k in keys if k is not None]
    res["score"] = float(np.mean(keys)) if keys else 0.0
    return res


# ============================================================================ checkpoint helpers
def atomic_save(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_local = Path("/tmp") / (path.name + ".tmp")
    torch.save(obj, tmp_local)
    tmp_remote = path.with_suffix(path.suffix + ".tmp")
    shutil.copy(tmp_local, tmp_remote)
    os.replace(tmp_remote, path)
    tmp_local.unlink(missing_ok=True)


class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone().float() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach().float(), alpha=1 - self.decay)
            else:
                self.shadow[k].copy_(v)

    def copy_to(self, model):
        model.load_state_dict({k: v.to(dtype=model.state_dict()[k].dtype) for k, v in self.shadow.items()})


# ============================================================================ train / eval
@torch.no_grad()
def predict(model, df, root, size, bs=32, tta=True, device="cuda", return_emb=False, workers=2):
    model.eval()
    dl = DataLoader(FundusDS(df, root, size, train=False), batch_size=bs, shuffle=False,
                    num_workers=workers, pin_memory=True)
    outs = {k: [] for k in HEADS}
    embs = []
    for x, _ in tqdm(dl, desc="predict", leave=False):
        x = x.to(device, non_blocking=True)
        views = [x, torch.flip(x, dims=[3])] if tta else [x]
        acc, eacc = None, None
        for v in views:
            with torch.autocast(device_type="cuda" if device != "cpu" else "cpu", dtype=torch.float16,
                                enabled=device != "cpu"):
                o, e = model(v, return_emb=True)
            o = {k: t.float() for k, t in o.items()}
            acc = o if acc is None else {k: acc[k] + o[k] for k in o}
            eacc = e.float() if eacc is None else eacc + e.float()
        for k in HEADS:
            outs[k].append((acc[k] / len(views)).cpu().numpy())
        if return_emb:
            embs.append((eacc / len(views)).cpu().numpy().astype(np.float16))
    res = {k: np.concatenate(v) for k, v in outs.items()}
    if return_emb:
        res["emb"] = np.concatenate(embs)
    return res


def train_model(key: str, data_dir="/content/veya_data", drive_dir="/content/drive/MyDrive/veya_v2",
                epochs: int | None = None, ckpt_minutes=12, workers=2, max_hours: float | None = None,
                quick_test=False, pretrained=True):
    cfg = dict(MODELS[key])
    if epochs:
        cfg["epochs"] = epochs
    device = "cuda" if torch.cuda.is_available() else "cpu"
    out_dir = Path(drive_dir) / "models" / key
    out_dir.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(Path(data_dir) / "labels.csv")
    tr = df[df.split == "train"].reset_index(drop=True)
    va = df[df.split == "val"].reset_index(drop=True)
    if quick_test:
        tr, va = tr.head(64), va.head(32)
        cfg.update(samples_per_epoch=32, bs=8, accum=1)
    print(f"[{key}] train={len(tr)} val={len(va)} size={cfg['size']} device={device}")

    pos_weight = {}
    for k in BINARY:
        y = tr[k][tr[k] >= 0]
        pos = max(1, int(y.sum()))
        pos_weight[k] = float(min(5.0, math.sqrt((len(y) - pos) / pos)))
    print("pos_weight:", pos_weight)

    model = build_model(key, pretrained=pretrained and not quick_test).to(device)
    if cfg.get("channels_last") and device == "cuda":
        model = model.to(memory_format=torch.channels_last)
    opt = torch.optim.AdamW(param_groups(model, cfg["lr"], layer_decay=cfg.get("layer_decay")), betas=(0.9, 0.999))
    scaler = torch.amp.GradScaler("cuda", enabled=device == "cuda")
    ema = EMA(model, decay=0.999 if not quick_test else 0.5)
    steps_per_epoch = cfg["samples_per_epoch"] // (cfg["bs"] * cfg["accum"])
    total_steps = steps_per_epoch * cfg["epochs"]
    warmup = max(1, int(0.03 * total_steps))

    def lr_factor(step):
        if step < warmup:
            return (step + 1) / warmup
        t = (step - warmup) / max(1, total_steps - warmup)
        return 0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * t))

    state = dict(epoch=0, step_in_epoch=0, global_step=0, best=-1.0, history=[])
    last = out_dir / "last.pt"
    if last.exists():
        ck = torch.load(last, map_location="cpu", weights_only=False)
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        scaler.load_state_dict(ck["scaler"])
        ema.shadow = {k: v.float().to(device) for k, v in ck["ema"].items()}
        state = ck["state"]
        print(f"▶ Продолжаю с эпохи {state['epoch']+1}, шаг {state['step_in_epoch']} (best={state['best']:.4f})")
    if state.get("done"):
        print("Модель уже обучена. Удалите last.pt чтобы обучить заново.")
        return out_dir

    weights = sample_weights(tr)
    ds = FundusDS(tr, data_dir, cfg["size"], train=True)
    t_start = time.time()
    last_ck = time.time()

    def save_last():
        atomic_save(dict(model=model.state_dict(), opt=opt.state_dict(), scaler=scaler.state_dict(),
                         ema=ema.shadow, state=state, cfg=cfg), last)

    while state["epoch"] < cfg["epochs"]:
        ep = state["epoch"]
        sampler = EpochSampler(weights, steps_per_epoch * cfg["bs"] * cfg["accum"], ep,
                               start=state["step_in_epoch"] * cfg["bs"] * cfg["accum"])
        dl = DataLoader(ds, batch_size=cfg["bs"], sampler=sampler, num_workers=workers, pin_memory=True,
                        drop_last=True, persistent_workers=False, prefetch_factor=4 if workers else None)
        model.train()
        pbar = tqdm(dl, desc=f"[{key}] epoch {ep+1}/{cfg['epochs']}", initial=state["step_in_epoch"] * cfg["accum"],
                    total=steps_per_epoch * cfg["accum"])
        run = {}
        for it, (x, y) in enumerate(pbar):
            x = x.to(device, non_blocking=True)
            if cfg.get("channels_last") and device == "cuda":
                x = x.to(memory_format=torch.channels_last)
            y = y.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda" if device == "cuda" else "cpu", dtype=torch.float16,
                                enabled=device == "cuda"):
                out = model(x)
            loss, parts = multitask_loss(out, y, pos_weight)
            scaler.scale(loss / cfg["accum"]).backward()
            for k, v in parts.items():
                run[k] = 0.98 * run.get(k, v) + 0.02 * v
            if (it + 1) % cfg["accum"] == 0:
                f = lr_factor(state["global_step"])
                for g in opt.param_groups:
                    g["lr"] = g["base_lr"] * f
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
                ema.update(model)
                state["global_step"] += 1
                state["step_in_epoch"] += 1
                pbar.set_postfix({k: f"{v:.3f}" for k, v in run.items()})
                if time.time() - last_ck > ckpt_minutes * 60:
                    save_last()
                    last_ck = time.time()
                if max_hours and time.time() - t_start > max_hours * 3600:
                    save_last()
                    print(f"⏸ Лимит времени {max_hours}ч — чекпоинт сохранён, перезапустите ячейку позже.")
                    return out_dir
        # ---- конец эпохи: валидация EMA-весов
        live = {k: v.detach().clone() for k, v in model.state_dict().items()}
        ema.copy_to(model)
        logits = predict(model, va, data_dir, cfg["size"], bs=cfg["bs"] * 2, tta=False, device=device, workers=workers)
        met = compute_metrics(logits, va[HEADS].values)
        met["epoch"] = ep + 1
        met["train_loss"] = {k: round(v, 4) for k, v in run.items()}
        state["history"].append(met)
        print(f"[{key}] epoch {ep+1}: " + ", ".join(f"{k}={v:.4f}" for k, v in met.items() if isinstance(v, float)))
        if met["score"] > state["best"]:
            state["best"] = met["score"]
            atomic_save(dict(model=model.state_dict(), cfg=cfg, metrics=met, key=key), out_dir / "best.pt")
            print(f"  ★ новый лучший score={met['score']:.4f} → best.pt")
        model.load_state_dict(live)
        state["epoch"] += 1
        state["step_in_epoch"] = 0
        save_last()
        last_ck = time.time()
        (out_dir / "history.json").write_text(json.dumps(state["history"], indent=1, default=float))
    state["done"] = True
    save_last()
    print(f"[{key}] обучение завершено. best score={state['best']:.4f}")
    return out_dir


def export_predictions(key: str, data_dir="/content/veya_data", drive_dir="/content/drive/MyDrive/veya_v2",
                       splits=("val", "test", "test_ext"), workers=2):
    """Предсказания лучшей модели (с TTA) + эмбеддинги → вход для MLP-«мозга»."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    out_dir = Path(drive_dir) / "models" / key
    ck = torch.load(out_dir / "best.pt", map_location="cpu", weights_only=False)
    model = build_model(key, pretrained=False)
    model.load_state_dict(ck["model"])
    model.to(device)
    df = pd.read_csv(Path(data_dir) / "labels.csv")
    pred_dir = Path(drive_dir) / "preds"
    pred_dir.mkdir(parents=True, exist_ok=True)
    for sp in splits:
        d = df[df.split == sp].reset_index(drop=True)
        if not len(d):
            continue
        r = predict(model, d, data_dir, ck["cfg"]["size"], bs=ck["cfg"]["bs"] * 2, tta=True,
                    device=device, return_emb=True, workers=workers)
        np.savez_compressed(pred_dir / f"{key}_{sp}.npz", image_id=d["image_id"].values, **r)
        met = compute_metrics(r, d[HEADS].values)
        print(f"[{key}] {sp}: " + ", ".join(f"{k}={v:.4f}" for k, v in met.items()))
    return pred_dir

In [ ]:
%%writefile /content/veya_v2/brain.py
"""VEYA v2 — «Мозг»: многослойная нейросеть, которая принимает решения по ответам всех моделей.

Каждая базовая модель («глаз») выдаёт:  вероятности по 4 задачам + эмбеддинг снимка.
Мозг (MLP 3 слоя) получает всё это вместе и выдаёт итоговые вероятности.
Обучается на val-части (её базовые модели не видели), проверяется на test и test_ext.

Плюс:
  * пороги по каждой болезни подбираются на val (а не тупо 0.5)
  * «зона сомнения»: если мозг не уверен — ответ «нужна проверка врача».
    Так модель смело говорит там, где уверена, и не врёт там, где не уверена.
"""
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, cohen_kappa_score, roc_auc_score
from sklearn.model_selection import GroupKFold

BINARY = ["dr", "glaucoma", "cataract"]
HEADS = ["dr_grade"] + BINARY
NAMES_RU = {"dr": "Диабетическая ретинопатия", "glaucoma": "Глаукома", "cataract": "Катаракта"}


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def softmax(x):
    e = np.exp(x - x.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)


# ============================================================================ features
def load_preds(pred_dir, models, split):
    out = {}
    for m in models:
        f = Path(pred_dir) / f"{m}_{split}.npz"
        if f.exists():
            out[m] = dict(np.load(f, allow_pickle=True))
    return out


class FeatureBuilder:
    """Признаки для мозга: вероятности каждой модели + PCA-сжатые эмбеддинги."""

    def __init__(self, models, n_pca=16):
        self.models = models
        self.n_pca = n_pca
        self.pca = {}
        self.mu = None
        self.sd = None

    def _raw(self, preds, fit=False):
        cols = []
        for m in self.models:
            p = preds[m]
            cols.append(softmax(p["dr_grade"].astype(np.float32)))
            cols.append(np.stack([p[k] for k in BINARY], 1).astype(np.float32))  # логиты
            e = p["emb"].astype(np.float32)
            if fit:
                self.pca[m] = PCA(min(self.n_pca, len(e) - 1, e.shape[1]), random_state=0).fit(e)
            cols.append(self.pca[m].transform(e).astype(np.float32))
        return np.concatenate(cols, 1)

    def fit_transform(self, preds):
        x = self._raw(preds, fit=True)
        self.mu, self.sd = x.mean(0), x.std(0) + 1e-6
        return (x - self.mu) / self.sd, self.base(preds)

    def transform(self, preds):
        return (self._raw(preds) - self.mu) / self.sd, self.base(preds)

    def base(self, preds):
        """Среднее мнение моделей (логиты) — мозг учится его ПОПРАВЛЯТЬ, а не угадывать с нуля."""
        g = np.mean([preds[m]["dr_grade"].astype(np.float32) for m in self.models], 0)
        b = np.mean([np.stack([preds[m][k] for k in BINARY], 1).astype(np.float32) for m in self.models], 0)
        return np.concatenate([g, b], 1)


# ============================================================================ brain network
class Brain(nn.Module):
    def __init__(self, d_in, hidden=(256, 128, 64), drop=0.3):
        super().__init__()
        layers, d = [], d_in
        for i, h in enumerate(hidden):
            layers += [nn.Linear(d, h), nn.BatchNorm1d(h), nn.GELU(), nn.Dropout(drop if i == 0 else drop / 2)]
            d = h
        self.body = nn.Sequential(*layers)
        self.grade = nn.Linear(d, 5)
        self.bin = nn.Linear(d, 3)
        for lin in (self.grade, self.bin):  # старт = ровно среднее мнение моделей
            nn.init.zeros_(lin.weight)
            nn.init.zeros_(lin.bias)

    def forward(self, x, base):
        h = self.body(x)
        return self.grade(h) + base[:, :5], self.bin(h) + base[:, 5:]


def brain_loss(grade, binl, y):
    loss = 0.0
    m = y[:, 0] >= 0
    if m.any():
        loss = loss + F.cross_entropy(grade[m], y[m, 0])
    for j in range(3):
        mm = y[:, j + 1] >= 0
        if mm.any():
            loss = loss + F.binary_cross_entropy_with_logits(binl[mm, j], y[mm, j + 1].float())
    return loss


def fit_brain(x, base, y, epochs=40, lr=1e-3, wd=1e-2, seed=0, device="cpu"):
    torch.manual_seed(seed)
    net = Brain(x.shape[1]).to(device)
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * (len(x) // 256 + 1))
    X = torch.tensor(x, dtype=torch.float32, device=device)
    B0 = torch.tensor(base, dtype=torch.float32, device=device)
    Y = torch.tensor(y, dtype=torch.long, device=device)
    g = torch.Generator().manual_seed(seed)
    for _ in range(epochs):
        net.train()
        perm = torch.randperm(len(X), generator=g)
        for i in range(0, len(X), 256):
            b = perm[i:i + 256]
            if len(b) < 2:
                continue
            xb = X[b] + 0.05 * torch.randn_like(X[b])  # шум = регуляризация
            gl, bl = net(xb, B0[b])
            loss = brain_loss(gl, bl, Y[b])
            opt.zero_grad()
            loss.backward()
            opt.step()
            sched.step()
    net.eval()
    return net


@torch.no_grad()
def brain_predict(nets, x, base, device="cpu"):
    X = torch.tensor(x, dtype=torch.float32, device=device)
    B0 = torch.tensor(base, dtype=torch.float32, device=device)
    gs, bs = [], []
    for n in nets:
        g, b = n(X, B0)
        gs.append(F.softmax(g, 1).cpu().numpy())
        bs.append(torch.sigmoid(b).cpu().numpy())
    return np.mean(gs, 0), np.mean(bs, 0)


# ============================================================================ thresholds + doubt zone
def best_threshold(y, p):
    """Порог, максимизирующий сбалансированную точность (чувствительность+специфичность)/2."""
    ts = np.unique(np.quantile(p, np.linspace(0.01, 0.99, 197)))
    best, bt = -1, 0.5
    for t in ts:
        ba = 0.5 * ((p[y == 1] >= t).mean() + (p[y == 0] < t).mean())
        if ba > best:
            best, bt = ba, t
    return float(bt)


def doubt_zone(y, p, thr, target=0.95, max_refer=0.25):
    """Ширина «зоны сомнения» вокруг порога: самая узкая, при которой точность на уверенных ≥ target."""
    for w in np.linspace(0, 0.45, 91):
        lo, hi = max(0.0, thr - w), min(1.0, thr + w)
        conf = (p < lo) | (p >= hi)
        if conf.mean() < 1 - max_refer:
            break
        if conf.sum() and ((p[conf] >= thr) == y[conf]).mean() >= target:
            return float(lo), float(hi)
    w = 0.0
    for w2 in np.linspace(0, 0.45, 91):
        lo, hi = max(0.0, thr - w2), min(1.0, thr + w2)
        if ((p < lo) | (p >= hi)).mean() < 1 - max_refer:
            break
        w = w2
    return float(max(0.0, thr - w)), float(min(1.0, thr + w))


def disease_report(y, p, thr, zone):
    m = y >= 0
    y, p = y[m].astype(int), p[m]
    if len(y) == 0 or y.min() == y.max():
        return None
    pred = (p >= thr).astype(int)
    conf = (p < zone[0]) | (p >= zone[1])
    tp = ((pred == 1) & (y == 1)).sum()
    tn = ((pred == 0) & (y == 0)).sum()
    return dict(
        n=int(len(y)), positives=int(y.sum()),
        auc=float(roc_auc_score(y, p)),
        accuracy=float((pred == y).mean()),
        sensitivity=float(tp / max(1, y.sum())),
        specificity=float(tn / max(1, (y == 0).sum())),
        balanced_accuracy=float(0.5 * (tp / max(1, y.sum()) + tn / max(1, (y == 0).sum()))),
        confident_share=float(conf.mean()),
        accuracy_confident=float((pred[conf] == y[conf]).mean()) if conf.any() else float("nan"),
    )


# ============================================================================ main
def run_brain(models=("effv2s", "convnext", "vit"), data_dir="/content/veya_data",
              drive_dir="/content/drive/MyDrive/veya_v2", folds=5, seeds=3, device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    pred_dir = Path(drive_dir) / "preds"
    labels = pd.read_csv(Path(data_dir) / "labels.csv").set_index("image_id")
    val = load_preds(pred_dir, models, "val")
    models = [m for m in models if m in val]
    print("Модели в ансамбле:", models)
    assert models, "Нет предсказаний. Сначала обучите модели и запустите export_predictions."
    ids = val[models[0]]["image_id"]
    for m in models:
        assert (val[m]["image_id"] == ids).all(), f"порядок снимков не совпадает у {m}"
    yv = labels.loc[ids, HEADS].values.astype(np.int64)
    groups = labels.loc[ids, "group"].values

    fb = FeatureBuilder(models)
    xv, bv = fb.fit_transform(val)
    print(f"Признаков для мозга: {xv.shape[1]}, обучающих снимков (val): {len(xv)}")

    # --- кросс-валидация на val: честные OOF-вероятности для подбора порогов
    oof_g = np.zeros((len(xv), 5))
    oof_b = np.zeros((len(xv), 3))
    for f, (tr, te) in enumerate(GroupKFold(folds).split(xv, groups=groups)):
        nets = [fit_brain(xv[tr], bv[tr], yv[tr], seed=s, device=device) for s in range(seeds)]
        g, b = brain_predict(nets, xv[te], bv[te], device)
        oof_g[te], oof_b[te] = g, b
    # сравнение с простым усреднением моделей
    avg_b = np.mean([sigmoid(np.stack([val[m][k] for k in BINARY], 1)) for m in models], 0)

    thresholds, zones, cmp, use_brain = {}, {}, {}, {}
    for j, k in enumerate(BINARY):
        m = yv[:, j + 1] >= 0
        if m.sum() == 0 or yv[m, j + 1].min() == yv[m, j + 1].max():
            continue
        cmp[k] = dict(brain_auc=roc_auc_score(yv[m, j + 1], oof_b[m, j]),
                      average_auc=roc_auc_score(yv[m, j + 1], avg_b[m, j]))
        # честный выбор по val: мозг используется только там, где он реально лучше усреднения
        use_brain[k] = bool(cmp[k]["brain_auc"] > cmp[k]["average_auc"])
        pv = oof_b[m, j] if use_brain[k] else avg_b[m, j]
        thresholds[k] = best_threshold(yv[m, j + 1], pv)
        zones[k] = doubt_zone(yv[m, j + 1], pv, thresholds[k])
    print("\nМозг vs простое усреднение (AUC на val, out-of-fold):")
    for k, v in cmp.items():
        print(f"  {NAMES_RU[k]:28s} мозг={v['brain_auc']:.4f}   среднее={v['average_auc']:.4f}   "
              f"→ {'мозг' if use_brain[k] else 'среднее'}")

    # --- финальный мозг на всём val
    final = [fit_brain(xv, bv, yv, seed=s, device=device) for s in range(seeds * 2)]

    report = {"models": list(models), "thresholds": thresholds, "doubt_zones": zones, "brain_vs_average": cmp,
              "use_brain": use_brain}
    for sp in ["test", "test_ext"]:
        P = load_preds(pred_dir, models, sp)
        if any(m not in P for m in models):
            continue
        ids_s = P[models[0]]["image_id"]
        ys = labels.loc[ids_s, HEADS].values.astype(np.int64)
        xs, bs_ = fb.transform(P)
        g, b = brain_predict(final, xs, bs_, device)
        avg_s = np.mean([sigmoid(np.stack([P[m][k] for k in BINARY], 1)) for m in models], 0)
        for j, k in enumerate(BINARY):
            if k in use_brain and not use_brain[k]:
                b[:, j] = avg_s[:, j]
        rep = {}
        for j, k in enumerate(BINARY):
            if k in thresholds:
                r = disease_report(ys[:, j + 1], b[:, j], thresholds[k], zones[k])
                if r:
                    rep[k] = r
        mg = ys[:, 0] >= 0
        if mg.sum() > 10:
            pg = g[mg].argmax(1)
            rep["dr_grade"] = dict(n=int(mg.sum()), accuracy=float(accuracy_score(ys[mg, 0], pg)),
                                   qwk=float(cohen_kappa_score(ys[mg, 0], pg, weights="quadratic")))
        # общая точность: доля правильных бинарных решений по всем размеченным болезням
        corr, tot, corr_c, tot_c = 0, 0, 0, 0
        for j, k in enumerate(BINARY):
            if k not in thresholds:
                continue
            mm = ys[:, j + 1] >= 0
            pr = b[mm, j] >= thresholds[k]
            corr += (pr == ys[mm, j + 1]).sum()
            tot += mm.sum()
            cf = (b[mm, j] < zones[k][0]) | (b[mm, j] >= zones[k][1])
            corr_c += (pr[cf] == ys[mm, j + 1][cf]).sum()
            tot_c += cf.sum()
        rep["overall"] = dict(accuracy=float(corr / max(1, tot)), decisions=int(tot),
                              accuracy_confident=float(corr_c / max(1, tot_c)),
                              confident_share=float(tot_c / max(1, tot)))
        # разбивка по источникам (видно, где модель переносится хуже)
        subs = labels.loc[ids_s, "subset"].values
        by_src = {}
        for s in np.unique(subs):
            ms = subs == s
            for j, k in enumerate(BINARY):
                if k not in thresholds:
                    continue
                r = disease_report(ys[ms, j + 1], b[ms, j], thresholds[k], zones[k])
                if r and r["n"] >= 30:
                    by_src[f"{s}/{k}"] = r
        rep["by_source"] = by_src
        report[sp] = rep
        print_report(sp, rep)
        if by_src:
            print("  по источникам:")
            for key, r in by_src.items():
                print(f"    {key:32s} n={r['n']:5d} AUC={r['auc']:.3f} bal_acc={r['balanced_accuracy']:.3f} "
                      f"sens={r['sensitivity']:.3f} spec={r['specificity']:.3f}")

    out = Path(drive_dir) / "brain"
    out.mkdir(parents=True, exist_ok=True)
    torch.save(dict(nets=[n.state_dict() for n in final], d_in=xv.shape[1], models=list(models),
                    pca={m: (fb.pca[m].components_, fb.pca[m].mean_) for m in models},
                    mu=fb.mu, sd=fb.sd, thresholds=thresholds, zones=zones, use_brain=use_brain), out / "brain.pt")
    (out / "report.json").write_text(json.dumps(report, indent=1, ensure_ascii=False, default=float))
    print(f"\nМозг и отчёт сохранены в {out}")
    return report


def print_report(split, rep):
    title = {"test": "ВНУТРЕННИЙ ТЕСТ (новые пациенты)", "test_ext": "ВНЕШНИЙ ТЕСТ (датасеты, которых модель не видела)"}
    print(f"\n===== {title.get(split, split)} =====")
    print(f"{'Болезнь':28s} {'n':>6s} {'AUC':>6s} {'Точн.':>6s} {'Чувств.':>7s} {'Спец.':>6s} {'Увер.%':>7s} {'Точн.увер.':>10s}")
    for k in BINARY:
        r = rep.get(k)
        if not r:
            continue
        print(f"{NAMES_RU[k]:28s} {r['n']:6d} {r['auc']:6.3f} {r['accuracy']:6.3f} {r['sensitivity']:7.3f} "
              f"{r['specificity']:6.3f} {100*r['confident_share']:6.1f}% {r['accuracy_confident']:10.3f}")
    if "dr_grade" in rep:
        r = rep["dr_grade"]
        print(f"Стадия ДР (0–4): accuracy={r['accuracy']:.3f}, QWK={r['qwk']:.3f} (n={r['n']})")
    o = rep["overall"]
    print(f"ОБЩАЯ ТОЧНОСТЬ: {100*o['accuracy']:.1f}%   |   на уверенных ответах: {100*o['accuracy_confident']:.1f}% "
          f"(уверенных {100*o['confident_share']:.0f}%, остальные → «нужен врач»)")

In [ ]:
%%writefile /content/veya_v2/export.py
"""VEYA v2 — экспорт обученной модели в ONNX для API (ONNX Runtime, без PyTorch на сервере).

В ONNX зашито: нормализация + TTA (обычный и зеркальный снимок, логиты усредняются) + softmax/sigmoid.
Вход:  image — float32 [1, 3, S, S], RGB, значения 0..255, снимок уже обрезан по кругу сетчатки (crop_fundus).
Выходы: dr_grade [1,5] (вероятности стадий 0–4), dr / glaucoma / cataract [1] (вероятности).

Рядом сохраняется model_config.json: пороги и «зона сомнения» по каждой болезни (подобраны на val),
и честные метрики на test / test_ext — чтобы API отвечал «здоров / болен / нужен врач».
"""
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from . import brain as B
from . import train as T

MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]
OUTPUTS = ["dr_grade", "dr", "glaucoma", "cataract"]


class OnnxWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.register_buffer("mean", torch.tensor(MEAN).view(1, 3, 1, 1) * 255)
        self.register_buffer("std", torch.tensor(STD).view(1, 3, 1, 1) * 255)

    def forward(self, image):
        x = (image - self.mean) / self.std
        a = self.model(x)
        b = self.model(torch.flip(x, dims=[3]))
        g = torch.softmax((a["dr_grade"] + b["dr_grade"]) / 2, dim=1)
        return (g,) + tuple(torch.sigmoid((a[k] + b[k]) / 2) for k in ("dr", "glaucoma", "cataract"))


def calibrate(key, drive_dir, labels: pd.DataFrame):
    """Пороги и зоны сомнения на val (TTA-логиты из export_predictions) + метрики на test/test_ext."""
    lab = labels.set_index("image_id")
    P = dict(np.load(Path(drive_dir) / "preds" / f"{key}_val.npz", allow_pickle=True))
    yv = lab.loc[P["image_id"]]
    thresholds, zones = {}, {}
    for k in B.BINARY:
        y = yv[k].values
        m = y >= 0
        p = B.sigmoid(P[k][m].astype(np.float64))
        thresholds[k] = B.best_threshold(y[m], p)
        zones[k] = B.doubt_zone(y[m], p, thresholds[k])
    metrics = {}
    for sp in ["test", "test_ext"]:
        f = Path(drive_dir) / "preds" / f"{key}_{sp}.npz"
        if not f.exists():
            continue
        Q = dict(np.load(f, allow_pickle=True))
        ys = lab.loc[Q["image_id"]]
        rep = {}
        for k in B.BINARY:
            r = B.disease_report(ys[k].values, B.sigmoid(Q[k].astype(np.float64)), thresholds[k], zones[k])
            if r:
                rep[k] = {n: round(float(v), 4) for n, v in r.items()}
        mg = ys["dr_grade"].values >= 0
        if mg.sum() > 10:
            from sklearn.metrics import accuracy_score, cohen_kappa_score
            pg = Q["dr_grade"][mg].argmax(1)
            rep["dr_grade"] = dict(n=int(mg.sum()), accuracy=round(float(accuracy_score(ys["dr_grade"].values[mg], pg)), 4),
                                   qwk=round(float(cohen_kappa_score(ys["dr_grade"].values[mg], pg, weights="quadratic")), 4))
        metrics[sp] = rep
    return thresholds, zones, metrics


def export_onnx(key="effv2s", data_dir="/content/veya_data", drive_dir="/content/drive/MyDrive/veya_v2",
                out_dir=None, opset=17, check_images=8):
    out_dir = Path(out_dir or Path(drive_dir) / "export")
    out_dir.mkdir(parents=True, exist_ok=True)
    ck = torch.load(Path(drive_dir) / "models" / key / "best.pt", map_location="cpu", weights_only=False)
    size = int(ck["cfg"]["size"])
    model = T.build_model(key, pretrained=False)
    model.load_state_dict(ck["model"])
    wrapper = OnnxWrapper(model).eval()

    onnx_path = out_dir / "model.onnx"
    dummy = torch.rand(1, 3, size, size) * 255
    torch.onnx.export(wrapper, dummy, str(onnx_path), input_names=["image"], output_names=OUTPUTS,
                      opset_version=opset, dynamo=False)
    print(f"ONNX сохранён: {onnx_path} ({onnx_path.stat().st_size / 1e6:.1f} МБ)")

    labels = pd.read_csv(Path(data_dir) / "labels.csv") if (Path(data_dir) / "labels.csv").exists() \
        else pd.read_csv(Path(drive_dir) / "data" / "labels.csv")
    thresholds, zones, metrics = calibrate(key, drive_dir, labels)

    # --- проверка: ONNX Runtime == PyTorch на настоящих снимках
    import onnxruntime as ort
    sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    cand = labels[labels.split.isin(["test", "test_ext"])]
    test = cand[[(Path(data_dir) / p).exists() for p in cand.path]].head(check_images)
    max_diff = 0.0
    for p in test.path:
        img = T.load_image(str(Path(data_dir) / p), size).astype(np.float32)
        x = img.transpose(2, 0, 1)[None]
        o = sess.run(OUTPUTS, {"image": x})
        with torch.no_grad():
            t = wrapper(torch.from_numpy(x))
        max_diff = max(max_diff, max(float(np.abs(a - b.numpy()).max()) for a, b in zip(o, t)))
    print(f"ONNX vs PyTorch на {len(test)} снимках: макс. расхождение вероятностей {max_diff:.2e}")

    config = {
        "format": "veya-v2",
        "model_name": f"veya-v2-{key}",
        "backbone": T.MODELS[key]["timm"],
        "image_size": size,
        "input": "RGB float32 0..255, NCHW, after crop_fundus (black borders cropped, centered square)",
        "outputs": {"dr_grade": "probs[5]", "dr": "prob", "glaucoma": "prob", "cataract": "prob"},
        "dr_grade_names": ["нет ДР", "лёгкая", "умеренная", "тяжёлая", "пролиферативная"],
        "thresholds": thresholds,
        "doubt_zones": zones,
        "metrics": metrics,
        "trained_metrics_val": {k: (round(float(v), 4) if isinstance(v, (int, float)) else v)
                                for k, v in ck.get("metrics", {}).items()},
        "onnx_vs_torch_max_diff": max_diff,
    }
    (out_dir / "model_config.json").write_text(json.dumps(config, indent=1, ensure_ascii=False))
    print(json.dumps({k: config[k] for k in ("thresholds", "doubt_zones")}, indent=1))
    for sp, rep in metrics.items():
        print(sp, {k: (v.get("auc"), v.get("accuracy")) if "auc" in v else v for k, v in rep.items()})
    print("Готово:", out_dir)
    return out_dir

## 1. Экспорт

In [ ]:
import tarfile, shutil, json
LOCAL = '/content/veya_data'
os.makedirs(LOCAL, exist_ok=True)
shutil.copy(f'{DRIVE}/data/labels.csv', f'{LOCAL}/labels.csv')
with tarfile.open(f'{DRIVE}/data/parts/idrid.tar') as t:   # немного снимков для проверки ONNX == PyTorch
    t.extractall(LOCAL)
from veya_v2 import export as E
KEY = 'effv2s'   # позже: лучшая модель / ансамбль
out = E.export_onnx(KEY, data_dir=LOCAL, drive_dir=DRIVE)
for f in sorted(os.listdir(out)): print(f, round(os.path.getsize(f'{out}/{f}') / 1e6, 1), 'МБ')

Файлы `model.onnx` и `model_config.json` лежат в `MyDrive/veya_v2/export/` — их кладём в `model_bundle/` репозитория API.